# Aprendizaje de Simbolo: `ParabolicMLPSymbol` con datos GRF (calor 2D)

Mismo experimento que `03_barrido_grf.ipynb` en $\mathbb{T}^2$: se aprende
$a_\theta(\mathbf{k})=\langle\mathbf{k}\rangle^m\big(c+(C-c)\,\sigma(g_\theta(\mathbf{k}/\langle\mathbf{k}\rangle))\big)$
($a_\theta(0)=0$) a partir de trayectorias de la ecuacion del calor, con referencia
$a(\mathbf{k})=|\mathbf{k}|^2$ (`HeatSymbol`, $n=2$, $m=2$, $\rho=n+1=3$ para el simbolo aprendido).

**Funcion de perdida** (la misma de 03)

$$\mathcal{L} = \lambda_{\text{traj}}\,\mathcal{L}_{\text{traj}} + \lambda_{\text{par}}\,\mathcal{L}_{\text{par}} + \lambda_{\text{class}}\,\mathcal{L}_{\text{class}}$$

- **Trayectoria:** $\mathcal{L}_{\text{traj}} = \mathbb{E}\big[|e^{-t a_\theta(\mathbf{k})}\hat u_0(\mathbf{k}) - \hat u_{\text{ref}}(\mathbf{k},t)|^2\big]$
- **Parabolicidad (Def. 2.3):** $\mathcal{L}_{\text{par}} = \mathbb{E}\big[\max\big(0,\ |(a_\theta(\mathbf{k})+\lambda)^{-1}| - \kappa(1+|(\mathbf{k},\lambda)|^2)^{-m/2}\big)^2\big]$,
  sobre $\mathrm{Re}(\lambda)\ge 0$, $|(\mathbf{k},\lambda)|\ge\omega$.
- **Clase de simbolo $S^{m,\rho}$:**
  $\mathcal{L}_{\text{class}} = \mathbb{E}_{\mathbf{k}}\Big[\frac{1}{|A_\rho|}\sum_{\alpha\in A_\rho}\max\big(0,\ \langle\mathbf{k}\rangle^{|\alpha|-m}\,|\Delta^\alpha_k a_\theta(\mathbf{k})| - C_\alpha\big)^2\Big]$,
  con $A_\rho=\{\alpha\in\mathbb{N}_0^n : |\alpha|\le\rho\}$ y $\langle\mathbf{k}\rangle=(1+|\mathbf{k}|^2)^{1/2}$.

# Setup
Configuracion identica a `03_barrido_grf.ipynb`; el codigo de cada seccion esta en
`parabolic_symbol.learning.experiment.GRFExperiment`.

- **Datos (`data/grf.py`):** toro $\mathbb{T}^2=[0,2\pi)^2$, $u(x)=\sum_k \hat u(k)e^{ik\cdot x}$,
  $u_0\sim\mathcal N\big(0,\ 7^4(-\Delta+7^2)^{-2.5}\big)$ en una malla de $128\times128$ puntos.
  Entrenamiento: $100$ condiciones iniciales, $t=0.05+0.1j$, $j=0,\dots,9$. Prueba: $20$ condiciones
  iniciales nuevas, $t=0.05+0.05j$, $j=0,\dots,19$.
- **Mascara de recuperabilidad:** se descartan los pares $(k,t)$ con $e^{-t\,a(k)}\le\varepsilon_{\texttt{float64}}$,
  y los $k$ con $|k|_\infty > k_{\text{cap}} = 32$. La ventana de $k$ que queda fija $k_{\max}$.
  Particion entrenamiento / validacion **por $\mathbf{k}$** (80% / 20% de los $k$ unicos).
- **$\kappa$:** $\kappa=$ `theoretical_kappa()`, con $c$ y $C$ estimados de los datos de entrenamiento
  (`estimate_symbol` + `estimate_bounds`, $\delta=0.1$) con $m=2$. Las $C_\alpha$ de $\mathcal{L}_{\text{class}}$
  salen del simbolo analitico en $|k_i|\le k_{\max}$, con margen $1.5$.
- **Barrido:** $\{2,3,4\}$ capas ocultas $\times$ $\{32,64,128\}$ neuronas $\times$ dropout $0.2$ en `none` /
  `middle` $\times$ weight decay $\{0,\ 10^{-4}\}$, con **early stopping** sobre la perdida total de validacion
  (trayectoria + parabolicidad + $\mathcal{L}_{\text{class}}$). Se elige la corrida de menor error relativo de validacion.
- `cuda` si esta disponible y `float64`.

In [3]:
from pathlib import Path

import torch

from parabolic_symbol.symbols.heat import HeatSymbol
from parabolic_symbol.symbols.fractional_laplacian import FractionalLaplacianSymbol
from parabolic_symbol.learning.experiment import GRFExperiment, summary_table
from parabolic_symbol.utils import default_device

device = default_device()
dtype = torch.float64
print(f"device = {device}  |  dtype = {dtype}")

RESULTS = Path("../results/04_calor_2d")   # tablas (CSV) y figuras (PNG)
RESULTS.mkdir(parents=True, exist_ok=True)

device = cuda  |  dtype = torch.float64


In [4]:
calor = GRFExperiment(
    reference=HeatSymbol(n=2), m=2.0, resolution=128, k_cap=32,
    results=RESULTS, dtype=dtype, device=device,
)

## 1. Datos de referencia

Ventana de $k$ resultante y numero de muestras por conjunto.

In [5]:
display(calor.load_data())

ENTRENAMIENTO
100 condiciones iniciales x 10 tiempos, resolucion 128x128: 2512 de 83200 pares (k, t) recuperables (eps=2.22e-16, |k|_inf <= 32) -> 251,200 muestras
  ventana de k recuperable: 1151 k, |k|_inf <= 26, |k| in [0, 26.8]
  max |k| por tiempo: t=0.05:26, t=0.15:15, t=0.25:12, t=0.35:10, t=0.45:8, t=0.55:8, t=0.65:7, t=0.75:6, t=0.85:6, t=0.95:6
PRUEBA
20 condiciones iniciales x 20 tiempos, resolucion 128x128: 4266 de 166400 pares (k, t) recuperables (eps=2.22e-16, |k|_inf <= 32) -> 85,320 muestras
  ventana de k recuperable: 1151 k, |k|_inf <= 26, |k| in [0, 26.8]
  max |k| por tiempo: t=0.05:26, t=0.10:18, t=0.15:15, t=0.20:13, t=0.25:12, t=0.30:10, t=0.35:10, t=0.40:9, t=0.45:8, t=0.50:8, t=0.55:8, t=0.60:7, t=0.65:7, t=0.70:7, t=0.75:6, t=0.80:6, t=0.85:6, t=0.90:6, t=0.95:6, t=1.00:6

k_cap = 32  ->  k_max = 26  (ventana recuperable: 1151 k, |k|_inf <= 26, |k| in [0, 26.8])


,muestras,k_unicos
conjunto,,
entrenamiento,200200,921
validacion,51000,230
prueba,85320,1151


## 2. $\kappa$ teorico y constantes $C_\alpha$

$c$ y $C$ se estiman solo con los datos de entrenamiento y fijan $\kappa=$ `theoretical_kappa()`. $\hat\kappa$ del simbolo de referencia se calcula solo como comparacion.

In [6]:
display(calor.estimate_constants())

estimate_symbol: 131,000 de 200,200 pares (k, t) recuperables (tol=1e-10); 591 de 921 k de entrenamiento
  ventana usada: 591 k, |k|_inf <= 21, |k| in [0, 21.4]
kappa_hat referencia = 2.0000
(c, C) de los datos  = (0.4500, 1.0976)   m = 2, ventana: 590 k, |k|_inf <= 21, |k| in [1, 21.4]
kappa teorico        = 3.1427


,C_alpha (x1.5)
alpha,
"(0, 0)",1.498891
"(0, 1)",3.354102
"(1, 0)",3.354102
"(0, 2)",3.000000
"(1, 1)",1.498891
"(2, 0)",3.000000
"(0, 3)",1.498891
"(1, 2)",1.498891
"(2, 1)",1.498891


## 3. Barrido de hiperparametros

$3$ profundidades $\times$ $3$ anchos $\times$ ubicacion del dropout $\times$ $2$ weight decays $=$ **30 entrenamientos** (con $2$ capas, `middle` equivale a `none` y no se repite).

In [7]:
calor.sweep()

[  1/30] 2x32  drop[-] wd=0e+00 | rel_l2=0.3439 | 464.9s


KeyboardInterrupt: 

## 4. Resultados del barrido

In [ ]:
barrido, por_eje = calor.sweep_results()
display(barrido)
for tabla in por_eje:
    display(tabla)

### Seleccion: la mejor aproximacion

Todas las corridas usan $\kappa=$ `theoretical_kappa()`, asi que se elige la de **menor error relativo de validacion**.

In [ ]:
display(calor.select())

### Metricas del modelo seleccionado

Trayectoria sobre las muestras de entrenamiento y validacion, y simbolo $a_\theta(\mathbf{{k}})$ frente a $|\mathbf{k}|^2$ sobre los $k$ de entrenamiento y los de validacion (que la red nunca vio).

In [ ]:
traj, sym = calor.metrics()
display(traj)
display(sym)

### Evaluacion en prueba

Error relativo $L^2$ sobre los pares $(k,t)$ recuperables de prueba y en espacio fisico (malla completa, todos los modos), por condicion inicial y tiempo.

In [ ]:
display(calor.test())

## 5. Curvas de entrenamiento

Perdida total de `evaluate_split` (incluye $\mathcal{{L}}_{{\text{{class}}}}$) y error relativo; la linea punteada marca la epoca que restauro el early stopping.

In [ ]:
calor.curves()

## 6. Simbolo aprendido vs. referencia

Simbolo frente a $|\mathbf{k}|^2$ en $|k_i|\le k_{\max}$: curva contra $|k|$, paridad y error absoluto contra $|k|$ (el simbolo es radial), y un mapa del error relativo sobre $(k_1,k_2)$.

In [ ]:
calor.symbol_figure()

## 7. Certificacion de parabolicidad

Rejilla independiente de la de entrenamiento: $|\lambda|$ log-espaciado en $[10^{-2},10^3]$ con $25$ angulos en $[-\pi/2,\pi/2]$, y $|k_i|\le K$ con $K=k_{\max}$ (dentro de la ventana), $2k_{\max}$ y $3k_{\max}$ (fuera).

In [ ]:
display(calor.certification())

## 8. Clase del simbolo: $S^{m,\rho}$

`SymbolClassVerifier` estima $\hat C_\alpha(K)$ en ventanas crecientes y la pendiente log-log. **Es evidencia en una ventana finita, no una prueba.**

In [ ]:
display(calor.symbol_class())